# Data Loading and Batching

## Objectives
- Understand Dataset and DataLoader
- Create custom datasets
- Handle train/val/test splits
- Use data transformations
- Optimize batch sampling

## Introduction
Efficient data loading is critical for deep learning. This notebook covers PyTorch's data loading utilities and best practices.

## What We're About to Do

The code below imports essential libraries. These libraries provide the foundational tools for tensor operations and neural network construction. Pay attention to what each import provides – understanding dependencies helps you know what's available for solving problems.


In [ ]:
# Import necessary libraries for tensor operations and deep learning
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
import pandas as pd

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")


In [ ]:
# Visualize the results
plt.figure(figsize=(12, 4))
plt.plot(range(len(result)), result, marker='o', linestyle='-', linewidth=2)
plt.title('Results Visualization', fontsize=14, fontweight='bold')
plt.xlabel('Index')
plt.ylabel('Value')
plt.grid(True, alpha=0.3)
plt.show()

print('Visualization shows the pattern and distribution of results.')


## 1. Built-in Dataset Classes

## What We're About to Do

The code below imports essential libraries. These libraries provide the foundational tools for tensor operations and neural network construction. Pay attention to what each import provides – understanding dependencies helps you know what's available for solving problems.


In [ ]:
# Import necessary libraries for tensor operations and deep learning
# TensorDataset - simple wrapping
from torch.utils.data import TensorDataset

X = torch.randn(100, 10)
y = torch.randint(0, 2, (100,))

dataset = TensorDataset(X, y)
print(f"Dataset length: {len(dataset)}")
print(f"First sample: {dataset[0]}")


## 2. Custom Dataset Class

In [ ]:
# Define a custom function with detailed implementation
class CustomDataset(Dataset):
# Iterate through batches of data
    """Custom dataset for flexibility"""
    def __init__(self, features, labels, transform=None):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
        self.transform = transform
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        x = self.features[idx]
        y = self.labels[idx]
        
        if self.transform:
            x = self.transform(x)
        
        return x, y

# Load iris dataset
iris = load_iris()
dataset = CustomDataset(iris.data, iris.target)

print(f"Dataset size: {len(dataset)}")
x, y = dataset[0]
print(f"Sample: x shape {x.shape}, y={y}")


## 3. DataLoader Basics

In [ ]:
# Execute code with detailed step-by-step process
# Create DataLoader
dataloader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
# Iterate through batches of data
    num_workers=0  # Set > 0 for multiprocessing
)

print(f"DataLoader length: {len(dataloader)} batches")

# Iterate through batches
# Iterate through batches of data
for batch_idx, (x_batch, y_batch) in enumerate(dataloader):
    print(f"Batch {batch_idx}: x shape {x_batch.shape}, y shape {y_batch.shape}")
    if batch_idx == 2:
        break


In [ ]:
# Execute code with detailed step-by-step process
# Different batch sizes
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Iterate through batches of data
for idx, batch_size in enumerate([8, 32, 128]):
    dl = DataLoader(dataset, batch_size=batch_size, shuffle=False)
# Iterate through batches of data
    axes[idx].bar(range(len(dl)), [x.shape[0] for x, _ in dl])
    axes[idx].set_title(f'Batch size: {batch_size}')
    axes[idx].set_xlabel('Batch Index')
    axes[idx].set_ylabel('Batch Size')
    axes[idx].set_ylim([0, batch_size + 10])

plt.tight_layout()
plt.show()


## The Training Process

This is the core learning loop. We'll forward-pass data through the model, compute loss, backpropagate gradients, and update parameters. This iterative process gradually improves the model.


In [ ]:
# Execute the training loop with proper tracking
## 4. Train/Val/Test Split

# Using random_split
total_size = len(dataset)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size]
)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
print(f"Total: {len(train_dataset) + len(val_dataset) + len(test_dataset)}")


## The Training Process

This is the core learning loop. We'll forward-pass data through the model, compute loss, backpropagate gradients, and update parameters. This iterative process gradually improves the model.


In [ ]:
# Execute the training loop with proper tracking
# Iterate through batches of data
# Create DataLoaders for each split
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")


## What We're About to Do

The code below imports essential libraries. These libraries provide the foundational tools for tensor operations and neural network construction. Pay attention to what each import provides – understanding dependencies helps you know what's available for solving problems.


In [ ]:
# Import necessary libraries for tensor operations and deep learning
## 5. Data Transformations

class Normalize:
    """Simple normalization transform"""
    def __init__(self, mean=None, std=None):
        self.mean = mean
        self.std = std
    
    def __call__(self, x):
        if self.mean is not None and self.std is not None:
            return (x - self.mean) / self.std
        return x

class AddNoise:
    """Add Gaussian noise"""
    def __init__(self, std=0.1):
        self.std = std
    
    def __call__(self, x):
        return x + torch.randn_like(x) * self.std

# Use transforms
from torchvision.transforms import Compose

transforms = Compose([
    Normalize(mean=0, std=1),
    AddNoise(std=0.05)
])

dataset_with_transform = CustomDataset(
    iris.data, iris.target, transform=transforms
)

print("Dataset with transforms created.")


In [ ]:
# Define a custom function with detailed implementation
## 6. Collate Function

def custom_collate_fn(batch):
    """Custom collate function"""
    x, y = zip(*batch)
    x = torch.stack(x)
    y = torch.tensor(y)
    
    # Add metadata
    return {
        'features': x,
        'labels': y,
        'batch_size': len(y)
    }

loader_custom = DataLoader(
    dataset,
    batch_size=16,
    collate_fn=custom_collate_fn
)

batch = next(iter(loader_custom))
print(f"Custom collate batch keys: {batch.keys()}")
print(f"Batch size: {batch['batch_size']}")


## The Training Process

This is the core learning loop. We'll forward-pass data through the model, compute loss, backpropagate gradients, and update parameters. This iterative process gradually improves the model.


In [ ]:
# Define a custom function with detailed implementation
## 7. DataPipeline Class

class DataPipeline:
    """Complete data pipeline"""
    def __init__(self, data, labels, batch_size=32, val_split=0.2, test_split=0.1):
        self.batch_size = batch_size
        
        # Create dataset
        self.dataset = CustomDataset(data, labels)
        
        # Split
        total_size = len(self.dataset)
        test_size = int(test_split * total_size)
        train_size = int((1 - val_split - test_split) * total_size)
        val_size = total_size - train_size - test_size
        
        self.train_ds, self.val_ds, self.test_ds = random_split(
            self.dataset, [train_size, val_size, test_size]
        )
        
        # Create loaders
        self.train_loader = DataLoader(
            self.train_ds, batch_size=batch_size, shuffle=True
        )
        self.val_loader = DataLoader(
            self.val_ds, batch_size=batch_size, shuffle=False
        )
        self.test_loader = DataLoader(
            self.test_ds, batch_size=batch_size, shuffle=False
        )
    
    def summary(self):
        print(f"Train: {len(self.train_ds)} ({len(self.train_loader)} batches)")
        print(f"Val: {len(self.val_ds)} ({len(self.val_loader)} batches)")
        print(f"Test: {len(self.test_ds)} ({len(self.test_loader)} batches)")

# Usage
pipeline = DataPipeline(iris.data, iris.target, batch_size=16)
pipeline.summary()


## The Training Process

This is the core learning loop. We'll forward-pass data through the model, compute loss, backpropagate gradients, and update parameters. This iterative process gradually improves the model.


In [ ]:
# Execute the training loop with proper tracking
## 8. Batch Visualization

# Get a batch
x_batch, y_batch = next(iter(pipeline.train_loader))

print(f"Batch shape: {x_batch.shape}")
print(f"Labels: {y_batch}")
print(f"Feature stats in batch:")
print(f"  Mean: {x_batch.mean(dim=0)}")
print(f"  Std: {x_batch.std(dim=0)}")
print(f"  Min: {x_batch.min(dim=0).values}")
print(f"  Max: {x_batch.max(dim=0).values}")


## What We're About to Do

The code below imports essential libraries. These libraries provide the foundational tools for tensor operations and neural network construction. Pay attention to what each import provides – understanding dependencies helps you know what's available for solving problems.


In [ ]:
# Import necessary libraries for tensor operations and deep learning
## 9. Memory and Performance

import time

batch_sizes = [8, 16, 32, 64, 128]
times = []

# Iterate through batches of data
for bs in batch_sizes:
    loader = DataLoader(dataset, batch_size=bs, shuffle=True)
    
    start = time.time()
# Iterate through batches of data
    for _ in loader:
        pass
    elapsed = time.time() - start
    times.append(elapsed)
    print(f"Batch size {bs}: {elapsed:.4f}s")

# Plot
plt.figure(figsize=(10, 6))
plt.plot(batch_sizes, times, marker='o', linewidth=2, markersize=8)
plt.xlabel('Batch Size')
plt.ylabel('Time (seconds)')
plt.title('Data Loading Speed vs Batch Size')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## The Training Process

This is the core learning loop. We'll forward-pass data through the model, compute loss, backpropagate gradients, and update parameters. This iterative process gradually improves the model.


## 🎯 Key Takeaways

✅ **Understanding fundamentals is crucial** – The concepts covered here form the foundation for all advanced deep learning techniques.

✅ **Each component has a specific purpose** – Whether it's data loading, model architecture, or optimization, every piece serves a distinct function in the pipeline.

✅ **Experimentation drives learning** – Don't just read the code; modify it, break it, and see what happens. That's how intuition develops.

✅ **Deep learning is iterative** – Success comes from systematically trying approaches, measuring results, and refining based on evidence.

✅ **Connect concepts, don't memorize** – Understanding how PyTorch tensors relate to autograd, which relates to neural networks, which connects to training loops, is far more valuable than memorizing individual APIs.

✅ **Performance matters in practice** – Once you understand the theory, optimizing for speed, memory, and scalability becomes crucial for real-world applications.


In [ ]:
# Set up the neural network model architecture
## Key Takeaways
- Dataset.__getitem__() and __len__() are the core interface
- DataLoader handles batching, shuffling, and multiprocessing
- Train/val/test split prevents overfitting and provides fair evaluation
- Transforms enable data augmentation and preprocessing
- Batch size affects training speed and memory usage

## Interview Q&A

**Q1: Why shuffle the training data but not validation/test?**
Shuffling training data helps the model learn diverse patterns and prevents order-dependent learning. Validation and test sets should be deterministic to ensure reproducibility.

**Q2: What's the difference between Dataset and DataLoader?**
Dataset defines how to access individual samples. DataLoader combines samples into batches and handles shuffling, multiprocessing, and pin_memory.

**Q3: How does num_workers affect performance?**
num_workers > 0 uses multiprocessing to load data in parallel, which can significantly speed up training. Use the number of CPU cores, but be careful with memory.

## References
- [PyTorch Data Loading](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)
- [Dataset and DataLoader API](https://pytorch.org/docs/stable/data.html)
